# Day 1 – Train two tiny language models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kouroshSA/nanoGPT-Shakespeare-Simpsons-Lab/blob/main/colab/Day1_Train_Models.ipynb)

**What you will do today:** teach two small GPT-style language models to write, one character at a time.
- **Model S** learns from the *complete works of Shakespeare* only.
- **Model M** learns from a *mix* of Shakespeare and *The Simpsons* dialogue.

Both models are the same size (about 19 million numbers, "parameters"). **The only difference is the text they read.**
Each model saves a snapshot ("checkpoint") every 1,000 steps, so on Day 2 you can see how the writing improves with training.

**Before you start:** Runtime ▸ Change runtime type ▸ **T4 GPU**. Then run the cells in order (▶ at the left of each cell).
Training takes a while (the progress is printed as it goes). Do not close the tab. Your snapshots are saved to your **Google Drive** (folder `nanoGPT_lab`), so Day 2 can find them.

*Made by Claude AI (Claude Sonnet 5.5), guided by K. Salehi-Ashtiani. Based on Andrej Karpathy's nanoGPT and Raj Shah's Simpsons edition.*

## 1. Setup

In [ ]:
# ── Setup (run this first) ───────────────────────────────────────────────────
import os, sys, subprocess

REPO_URL = "https://github.com/kouroshSA/nanoGPT-Shakespeare-Simpsons-Lab"
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 1. get the code (in Colab) -------------------------------------------------
if IN_COLAB and not os.path.exists("train.py"):
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/lab"], check=True)
    os.chdir("/content/lab")
print("working in:", os.getcwd())

# 2. where do the results live?  Colab erases everything when the session ends, so we use Google Drive ---
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    LAB_DIR = "/content/drive/MyDrive/nanoGPT_lab"
else:
    LAB_DIR = os.path.abspath("lab_runs")            # running on your own computer
os.makedirs(LAB_DIR, exist_ok=True)
print("results folder:", LAB_DIR)

def run(cmd):
    """Run a command and show its output live."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU found. In Colab: Runtime ▸ Change runtime type ▸ T4 GPU. (Training on a CPU is far too slow.)")

## 2. Prepare the text
This cleans the two books of text, mixes them, and turns every character into a number.
Both models use the **same table of 87 characters**, which is what lets us swap models on Day 2.

In [ ]:
run("python data/shakespeare_complete_char/prepare.py")
run("python data/shakespeare_simpsons_mix/prepare.py")

## 3. Train Model S (Shakespeare only)
Ten thousand training steps. Watch the **val loss** (how wrong the model is on text it has never seen): lower is better.
It starts near 4.5 (a coin-toss between 87 characters) and should fall to about 1.1.

If your session disconnects, run **Setup** again and then the cell below again with `RESUME = True`.

In [ ]:
RESUME = False                     # True = continue from the newest checkpoint after a disconnect
EXTRA = ""                         # (instructors: extra train.py options go here)
cmd = (f"python train.py config/train_shakespeare_complete_char.py --out_dir={LAB_DIR}/shakespeare "
       f"--compile=False --ckpt_save_optimizer=False {EXTRA}")
if RESUME:
    cmd += " --init_from=resume"
run(cmd)

### The learning curve of Model S
Blue = loss on the text it trained on; orange = loss on text it never saw. If orange stops falling (or rises) while blue keeps falling,
the model has started to **memorise** instead of learn.

In [ ]:
from IPython.display import Image, display
display(Image(f"{LAB_DIR}/shakespeare/loss_curve.png"))

## 4. Train Model M (Shakespeare + Simpsons)
Same settings and same size, different text.

In [ ]:
RESUME = False
cmd = (f"python train.py config/train_shakespeare_simpsons_mix.py --out_dir={LAB_DIR}/mix "
       f"--compile=False --ckpt_save_optimizer=False {EXTRA}")
if RESUME:
    cmd += " --init_from=resume"
run(cmd)

In [ ]:
display(Image(f"{LAB_DIR}/mix/loss_curve.png"))

## 5. Check that everything is saved
You should see **10 checkpoints in each folder** (`ckpt_001.pt` … `ckpt_010.pt`), about 77 MB each.

In [ ]:
for m in ["shakespeare", "mix"]:
    folder = f"{LAB_DIR}/{m}"
    files = sorted(f for f in os.listdir(folder) if f.startswith("ckpt_") and f.endswith(".pt")) if os.path.isdir(folder) else []
    print(f"{m:12s} {len(files)} checkpoints", "✅" if len(files) == 10 else "⚠️ (expected 10 – re-run the training cell with RESUME = True)")

## 6. Look at one early and one late snapshot (optional, 1 minute)
Just to see that something was learned. The real exploration is on **Day 2**.

In [ ]:
from lab_tools import checkpoints, generate
for m in ["shakespeare"]:
    cks = checkpoints(f"{LAB_DIR}/{m}")
    for p in (cks[0], cks[-1]):
        print("=" * 60, os.path.basename(p)); print(generate(p, "HAMLET.\n", n_chars=300, temperature=0.8))

**Done for today.** Your checkpoints are in Google Drive, in the folder `nanoGPT_lab`. See you on Day 2.